## CuPy: A GPU-Accelerated Drop-In Replacement for NumPy

**CuPy** is a library that mirrors NumPy’s interface but executes computations on the GPU, providing an almost seamless way to accelerate Python code.

Like NumPy, CuPy provides three fundamental components:

1. A **multidimensional array object** stored in GPU memory.  
2. A **universal function (ufunc)** system that supports broadcasting and executes element-wise operations in parallel on the GPU.  
3. A **comprehensive collection of array operations and mathematical functions**, implemented in CUDA for high-performance GPU execution.

One of CuPy’s key strengths is its ability to serve as a *drop-in replacement* for NumPy.  
This makes it possible to write **agnostic code** that runs on either the CPU or the GPU, depending on the available hardware, with minimal or no code modifications.

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

## Basic Operations with CuPy (Comparing with NumPy)

Let's start by comparing **NumPy** and **CuPy** for a few basic operations such as vector addition and matrix multiplication.

In [ ]:
# Vector sum using NumPy (CPU)
x_cpu = np.linspace(0, 100, 20)
y_cpu = np.linspace(10, 200, 20)
z_cpu = x_cpu + y_cpu
z_cpu

In [ ]:
# Matrix-matrix multiplication using NumPy (CPU)
x_cpu = np.random.random(1_000).reshape(20, 50)
y_cpu = np.random.random(1_000).reshape(50, 20)
z_cpu = np.dot(x_cpu, y_cpu)
z_cpu

To perform the same operations with **CuPy**, we simply need to replace the `numpy` import with `cupy`.  
This enables the same code to execute on the GPU instead of the CPU, with minimal modification.

In [ ]:
# Vector sum using CuPy (GPU)
x_gpu = cp.linspace(0, 100, 20)
y_gpu = cp.linspace(10, 200, 20)
z_gpu = x_gpu + y_gpu
z_gpu

In [ ]:
# Matrix-matrix multiplication using CuPy (GPU)
x_gpu = cp.random.random(1_000).reshape(20, 50)
y_gpu = cp.random.random(1_000).reshape(50, 20)
z_gpu = cp.dot(x_gpu, y_gpu)
z_gpu

Unlike (yet similarly to) NumPy’s `numpy.ndarray` objects, CuPy arrays are represented as `cupy.ndarray`.  

By default, CuPy **infers the data type** automatically.  
However, it may not always choose the most efficient format for your workload.  
It is therefore a good practice to **explicitly specify the data type** (e.g., `float32` or `float64`) to achieve optimal performance and memory utilization, especially for large datasets or GPU-bound operations.

In [ ]:
# Checking the type, data type, and shape of the NumPy array
print(type(z_cpu))     # Type of the array
print(z_cpu.dtype)     # Data type of the elements in the array
print(z_cpu.shape)     # Shape of the array

In [ ]:
# Checking the type, data type, shape, and device of the CuPy array
print(type(z_gpu))       # Type of the array
print(z_gpu.dtype)       # Data type of the elements in the array
print(z_gpu.shape)       # Shape of the array
print(z_gpu.device)      # Device where the array is allocated (GPU)

All operations executed with CuPy involve a certain amount of overhead in execution time.  
This overhead primarily comes from the time required to compile the code into CUDA kernels and to transfer data between the host (CPU) and the device (GPU).

Although memory management in CuPy is handled automatically, it may not always achieve optimal performance for all workloads.  
However, you can explicitly control **data transfers** between the CPU and GPU when necessary to improve efficiency or manage resources.

In [ ]:
# Create a NumPy array on the host and transfer it to the device (GPU)
a_cpu = np.array([0, 1, 2, 3, 4, 5])
a_gpu = cp.asarray(a_cpu)  # Transfer to GPU

In [ ]:
# Perform a computation on the CuPy array on the device (GPU)
b_gpu = cp.exp(a_gpu.reshape(2, 3))  # Calculate the exponential of each element after reshaping

In [ ]:
# Print the results
#
# This triggers a host-to-device transfer
# But the host memory is discarded after display
b_gpu

In [ ]:
# Copy the device data back to the host (CPU)
b_cpu = b_gpu.get()  # Transfer the data from GPU to CPU
b_cpu

## Agnostic Code

Thanks to the CuPy team's effort in providing a **1-to-1 mapping** of the NumPy API, one of the key advantages of CuPy is the ability to write functions that can be executed interchangeably on either the CPU or the GPU.

CuPy also provides a mechanism to **identify the array type** at runtime, allowing us to write **device-agnostic code**.  
This means a single function can operate transparently on both NumPy (CPU) and CuPy (GPU) arrays, automatically selecting the appropriate backend based on where the data resides.

In [ ]:
# Agnostic function implementation
def softplus(x):
    # Determine whether the input array is a NumPy or CuPy array
    # `xp` will be set to `cp` if x is a CuPy array, or `np` if it is a NumPy array
    xp = cp.get_array_module(x)
    print("Using:", xp.__name__)  # Display the backend library being used

    # Compute the Softplus function in a backend-agnostic way
    return xp.maximum(0, x) + xp.log1p(xp.exp(-abs(x)))

In [ ]:
# Create a NumPy array on the host (CPU) and transfer it to the device (GPU)
x_cpu = np.random.random(10_000)      # Random array on the CPU
x_gpu = cp.asarray(x_cpu)             # Transfer the array to the GPU

In [ ]:
# Apply the Softplus function on the CPU (NumPy)
result_cpu = softplus(x_cpu)
result_cpu

In [ ]:
# Apply the Softplus function on the GPU (CuPy)
result_gpu = softplus(x_gpu)
result_gpu

## Embedded Benchmarking

CuPy includes a built-in profiler that simplifies the creation and management of all `cuda.Event` objects needed for measuring execution time on the GPU.  
This makes it easy to benchmark the performance of GPU computations directly within your Python code or notebooks.

In [ ]:
# Import the CuPy profiler for benchmarking
# (cupyx is the CuPy extension module that includes additional utilities)
from cupyx.profiler import benchmark

In [ ]:
# Benchmark the softplus function on the CPU
cpu_bench = benchmark(softplus, (x_cpu,), n_repeat=10)  # Repeat the benchmark 10 times
print(cpu_bench)

In [ ]:
# Benchmark the softplus function on the GPU
gpu_bench = benchmark(softplus, (x_gpu,), n_repeat=10)  # Repeat the benchmark 10 times
print(gpu_bench)  # Display the benchmark results

## User-Defined Kernels

CuPy provides three types of **CUDA kernel definitions**:
- **Elementwise kernels**
- **Reduction kernels**
- **Raw kernels**

### Elementwise Kernels

Elementwise kernels perform operations that are applied independently to each element of one or more input arrays. These operations are executed in parallel across all data elements, allowing efficient, large-scale computations.

Conceptually, this is similar to the `@vectorize` or `@guvectorize` functions we encountered when using Numba with CUDA.

The definition of an elementwise kernel in CuPy includes four parts:
1. The list of input arguments.
2. The list of output arguments.
3. The body of the kernel (the computation to perform).
4. The name of the kernel.

In [ ]:
# Define an elementwise kernel using CuPy
kernel = cp.ElementwiseKernel(
    'float32 x, float32 y',             # Input arguments
    'float32 z',                        # Output argument
    '''if (x - 2 > y) { z = x * y; }     
    else { z = x + y; }''',             # Kernel body (executed on each thread)
    'elemwise_kernel'                   # Kernel name
)

In [ ]:
# Create input CuPy arrays
x = cp.arange(6, dtype='float32').reshape(2, 3)
y = cp.arange(3, dtype='float32')

In [ ]:
# Benchmark the elementwise kernel using CuPy's profiler
kernel_bench = benchmark(kernel, (x, y), n_repeat=10)
print(kernel_bench)

In [ ]:
# Execute the kernel and retrieve the result
z = kernel(x, y)
result = z.get()  # Transfer result from GPU to host
result

The same kernel, which is currently defined for `float32` arrays, can be generalized to operate on **arbitrary data types**.  
By defining the inputs and outputs with the generic type `T`, the appropriate data type is determined at compile time. This approach improves reusability and flexibility.

In [ ]:
# Define a generic type elementwise kernel
kernel_gtype = cp.ElementwiseKernel(
    'T x, T y',                          # Generic input arguments
    'T z',                               # Generic output argument
    '''if (x - 2 > y) { z = x * y; }     
    else { z = x + y; }''',              
    'elemwise_kernel_generic_type'       # Kernel name
)

In [ ]:
# Create integer input arrays and execute the generic kernel
x = cp.arange(6, dtype='int32').reshape(2, 3)
y = cp.arange(3, dtype='int32')

# Apply the generic type kernel
z = kernel_gtype(x, y)
result_generic = z.get()
result_generic

### Reduction Kernels

Reduction kernels perform operations that combine multiple elements of one or more input arrays into a single output by applying a **reduction operation**.  
Common examples include computing the sum, minimum, maximum, or mean of array elements.

Reduction kernels are particularly useful for efficiently computing global aggregates or statistics over large datasets, as the computation is parallelized across all GPU threads.

The definition of a reduction kernel in CuPy consists of the following components:
1. **Identity Value**: The initial value for the reduction process (e.g., `0` for sum, `-inf` for max).
2. **Mapping Expression**: The expression used to preprocess each input element before aggregation.
3. **Reduction Expression**: The operation that combines multiple mapped values (using the special variables `a` and `b` as operands).
4. **Post Mapping Expression**: A final transformation applied to the reduced value, using the special variable `a`. The result must be written to the output parameter.

In [ ]:
# Define a reduction kernel for computing the Euclidean (L2) norm using CuPy
l2norm_kernel = cp.ReductionKernel(
    'T x',              # Input parameter: array elements (generic type T)
    'T y',              # Output parameter: single scalar of type T
    'x * x',            # Mapping expression: square each input element 
    'a + b',            # Reduction expression: accumulate partial sums
    'y = sqrt(a)',      # Post-reduction mapping: take the square root of the accumulated sum
    '0',                # Identity value for the reduction (starting value)
    'l2norm'            # Name of the kernel
)

In [ ]:
# Create a CuPy array and compute the L2 norm using the reduction kernel
x = cp.arange(10, dtype=np.float32).reshape(2, 5)  # 2x5 array of float32

# Compute L2 norm along axis 1
l2norm_result = l2norm_kernel(x, axis=1)

# Print the results
l2norm_result_host = l2norm_result.get()
l2norm_result_host

### Raw Kernels (The Good Old CUDA-C Kernels...)

With **raw kernels**, we can define and launch CUDA kernels directly from raw CUDA source code.

Much like with **Numba+CUDA**, this approach bridges Python and CUDA-C, allowing us to leverage the same CUDA-C functions we have previously written — but now callable from Python.  

This enables a hybrid workflow: the majority of the codebase can be written in Python (and accelerated with CuPy), while specific performance-critical parts are expressed as low-level CUDA kernels or imported from external CUDA-C libraries.

In this example, we define and launch a **raw CUDA kernel** for matrix multiplication.

In [ ]:
# Define a raw CUDA kernel for matrix multiplication using CuPy
custom_kernel = cp.RawKernel(r'''
    extern "C" __global__ 
    void naiveMatrixMultiplication(const float* M, const float* N, float* P, const int width) {
        // Calculate the thread's position in the grid
        int row = blockIdx.y * blockDim.y + threadIdx.y;
        int col = blockIdx.x * blockDim.x + threadIdx.x;

        // Each thread computes one element of the result matrix
        if (row < width && col < width) {
            float sum = 0.0;
            // Perform dot product for row of M and column of N
            for (int k = 0; k < width; ++k) {
                sum += M[row * width + k] * N[k * width + col];
            }
            P[row * width + col] = sum;
        }
    }
    ''',
    'naiveMatrixMultiplication')  # Kernel name

In [ ]:
# Define matrix dimensions and initialize input and output arrays
width = 2048  # Square matrix size
M = cp.random.random((width, width), dtype='float32')
N = cp.random.random((width, width), dtype='float32')
P = cp.zeros_like(M)  # Output matrix

In [ ]:
# Define execution configuration
threads_per_block = (32, 32)
blocks = (
    (width + threads_per_block[0] - 1) // threads_per_block[0],
    (width + threads_per_block[1] - 1) // threads_per_block[1]
)

print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks)

In [ ]:
# Launch the CUDA kernel
custom_kernel(blocks, threads_per_block, (M, N, P, width))

# Retrieve result from GPU to host
P_host = P.get()

In [ ]:
# Benchmark the raw CUDA kernel
kernel_bench = benchmark(custom_kernel, (blocks, threads_per_block, (M, N, P, width)), n_repeat=5)
print(kernel_bench)

Clearly, this example is somewhat **unfair** when compared directly to CuPy’s built-in `cp.dot()` operation:

In [ ]:
# Define a CuPy-based reference implementation using CuPy's dot product
def cp_matmul(M, N):
    return cp.dot(M, N)

In [ ]:
# Benchmark the CuPy implementation
cp_matmul_bench = benchmark(cp_matmul, (M, N), n_repeat=5)
print(cp_matmul_bench)

In [ ]:
# Verify that both results are numerically consistent
P_cp = cp_matmul(M, N)
is_close = np.allclose(P, P_cp)
is_close

CuPy’s `dot` function is a high-level wrapper around **cuBLAS**, NVIDIA’s highly optimized linear algebra library.  
It performs extensive optimizations under the hood (tiling, caching, etc), none of which are implemented in our simple *naive* Raw Kernel example.

For most users and applications, **CuPy’s built-in functions** (such as `cp.dot`, `cp.matmul`, or `cp.tensordot`) should be preferred as they leverage NVIDIA’s mature, hardware-optimized libraries and deliver near-maximum performance.

However, **Raw Kernels** are invaluable when:
- You need **custom operations** that cannot be expressed through existing CuPy/NumPy functions.
- You want to **experiment with low-level CUDA optimization techniques**.
- You need to integrate **legacy or specialized CUDA-C kernels** directly in Python.

## Advanced Algebraic and Scientific Applications Made Simple

Most of the algebraic functionalities from NumPy, as well as some from SciPy (though not all; please check the documentation for details), are included in CuPy's library of rewritten CUDA kernels that operate on CuPy inputs.

* [Reference of NumPy routines included in CuPy](https://docs.cupy.dev/en/stable/reference/routines.html)
* [Reference of SciPy routines included in CuPy](https://docs.cupy.dev/en/stable/reference/scipy.html)
* [An extremely useful comparison between NumPy and CuPy](https://docs.cupy.dev/en/stable/reference/comparison.html)

### Algebraic functions

In [ ]:
# Perform Singular Value Decomposition (SVD) using NumPy
x_cpu = np.random.random((1000, 1000))  # Generate a random 1000x1000 matrix
u, s, v = np.linalg.svd(x_cpu)          # Compute the SVD, resulting in U, singular values S, and V

In [ ]:
# Perform Singular Value Decomposition (SVD) using CuPy
x_gpu = cp.asarray(x_cpu)               # Convert the NumPy array to a CuPy array
u_cp, s_cp, v_cp = cp.linalg.svd(x_gpu) # Compute the SVD on the GPU, resulting in U, S, and V

### Fitting and evaluating functions

In [ ]:
# Generate noisy data for polynomial fitting
x = np.linspace(0, 10, 100)
y_true = 2 * np.sin(x) + 0.5 * x
noise = np.random.normal(0, 0.5, size=x.shape)
y = y_true + noise

# Perform polynomial fitting with NumPy
degree = 5
coeffs = np.polyfit(x, y, degree)       # Fit a polynomial of specified degree
y_pred = np.polyval(coeffs, x)          # Evaluate the polynomial at x positions

# Plot data and fitted polynomial
plt.figure(figsize=(8, 4))
plt.plot(x, y, '.', label='Data')
plt.plot(x, y_pred, '-', label='Fit')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc='best')
plt.show()

In [ ]:
# Convert the NumPy arrays to CuPy arrays
x_gpu = cp.asarray(x)
y_gpu = cp.asarray(y)

# Perform polynomial fitting with CuPy
degree = 5
coeffs_cp = cp.polyfit(x_gpu, y_gpu, degree)
y_pred_cp = cp.polyval(coeffs_cp, x_gpu)

# Plot CPU and GPU fits for comparison
plt.figure(figsize=(8, 4))
plt.plot(x, y, '.', label='Data')
plt.plot(x, cp.asnumpy(y_pred), '-', label='Fit (CPU)')
plt.plot(x, cp.asnumpy(y_pred_cp), '-', label='Fit (GPU)')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc='best')
plt.show()

### SciPy-equivalent functionalities

In [ ]:
# 2D Convolution Example (Using delta Functions smeared by a Gaussian Filter)

# Create an image with repeated delta functions
deltas = np.zeros((2048, 2048))
deltas[8::16, 8::16] = 1  # Set every 16th pixel starting from (8,8)

# Plot a zoomed-in section of the delta grid
plt.imshow(deltas[0:200, 0:200])
plt.colorbar()
plt.show()

In [ ]:
# Create a Gaussian filter
x, y = np.meshgrid(np.linspace(-2, 2, 15), np.linspace(-2, 2, 15))
sigma = 1.0
muu = 0.0
dst = np.sqrt((x - muu) ** 2 + (y - muu) ** 2)
gauss = np.exp(-(dst ** 2) / (2.0 * sigma ** 2))

# Plot the Gaussian filter
plt.imshow(gauss)
plt.colorbar()
plt.show()

In [ ]:
# Transfer arrays to the GPU
deltas_gpu = cp.asarray(deltas)
gauss_gpu = cp.asarray(gauss)

In [ ]:
# Perform 2D convolution using CuPy's SciPy-compatible module
# (Equivalent in CuPy to the host code: `from scipy.signal import convolve2d`)
from cupyx.scipy.signal import convolve2d

convolved_img_gpu = convolve2d(deltas_gpu, gauss_gpu)

In [ ]:
# Transfer the result back to the host
convolved_img = convolved_img_gpu.get()

In [ ]:
# Plot a zoomed-in section of the convolved image
plt.imshow(convolved_img[0:200, 0:200])
plt.colorbar()
plt.show()